# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(metadata.name + ": " + metadata.description)
print("\nLicense:", metadata.license)
print("\nKeywords:", ', '.join(metadata.keywords))

## 2. Data Overview
Review available record sets, fields, and their IDs.

We enumerate the available record sets and their fields, referencing them by their unique `@id`s.

In [ ]:
# List all available record sets and their fields, referencing by @id
record_sets = list(dataset.record_sets)
print(f"Available record sets ({len(record_sets)} total):\n")
for rs in record_sets:
    print(f"  RecordSet name: {rs.name} | @id: {rs.id}")
    print("    Fields:")
    for f in rs.fields:
        print(f"      - {f.name} | @id: {f.id} | dataType: {f.data_type}")
    print("")

## 3. Data Extraction
Load data from specific record set(s) into a DataFrame for analysis.

Below, we will select the main survey record set (typically the largest, with fields for demographic details and survey scores) for demonstration. **All references use the `@id` of each record set.**

In [ ]:
# Example: Select main survey record set by @id
# (Replace this ID if the overview above shows a different @id or name you wish to analyze)
main_record_set_id = None
for rs in record_sets:
    if 'survey' in rs.name.lower() or 'main' in rs.name.lower() or 'response' in rs.name.lower():
        main_record_set_id = rs.id
        break
if main_record_set_id is None and record_sets:
    main_record_set_id = record_sets[0].id  # fallback to first

# You may add more record set @ids for further analysis
record_set_ids = [main_record_set_id]

dataframes = {}
for rid in record_set_ids:
    records = list(dataset.records(record_set=rid))
    df = pd.DataFrame(records)
    dataframes[rid] = df
    print(f"Loaded {len(df)} records for record set @id: {rid}")
    print("Columns available:", df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For this demonstration, suppose we are interested in scores for the Generalized Anxiety Disorder (GAD-7) assessment.

In [ ]:
# Identify a numeric field related to GAD-7 score from field ids or columns
# (Update this if overview shows an alternate @id)
df = dataframes[main_record_set_id]

# Try to heuristically pick a column related to Gad7/PHQ9
candidate_numeric_fields = [col for col in df.columns if 'gad' in col.lower() and 'score' in col.lower()] or [col for col in df.columns if col.lower() == 'gad7_score']
if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]
else:
    # Fallback to any float/integer column
    num_cols = df.select_dtypes(include=['float', 'int']).columns.tolist()
    numeric_field = num_cols[0] if num_cols else df.columns[0]

print(f"Using numeric field for filtering: {numeric_field}")

# Filtering by a threshold
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalize the numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping by a demographic field, e.g., 'gender' or 'age_group'
group_field = None
for possible in ['gender', 'sex', 'Gender', 'Sex', 'age_group', 'AgeGroup', 'age']:
    if possible in df.columns:
        group_field = possible
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
    display(grouped_df)
else:
    print("No suitable group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We will plot the distribution of the GAD-7 scores and, if possible, compare by group (e.g., gender).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of GAD-7 Score
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), bins=10)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot by group field, if available
if group_field:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides detailed survey results with fields for both demographic data and mental health assessments (e.g., GAD-7).
- Data can be easily accessed and processed using the Croissant schema and the `mlcroissant` library.
- Preliminary filters and plots highlight the range of GAD-7 scores and their distribution across demographic groups.
- For a deeper analysis, repeat these steps for other fields (e.g., PHQ-9, PSQ) or other record sets using their unique `@id`s.
For more advanced processing, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/python/intro/) for further recipes.